# 📊 0DTE Options Trading Setup — Report Generator

Generates a full intraday options analysis report (HTML) for **any US stock ticker**.

Covers: current price & trend · key levels · IV estimate · PCR sentiment · options chain snapshot · strategy recommendation · entry/target/stop · risk & invalidation.

| Step | Action |
|------|--------|
| **1** | Run **Cell 1** once per session to install packages |
| **2** | Run **Cell 2** to load all functions |
| **3** | Run **Cell 3**, type your ticker, click **Generate Report** |

> **Data source:** [yfinance](https://github.com/ranaroussi/yfinance) — free, no API key needed.

In [ ]:
# ── CELL 1 · Install dependencies (run once per session) ─────────────────────
!pip install -q yfinance pandas ipywidgets
print('✅ Dependencies ready')

In [ ]:
# ── CELL 2 · Load all functions ───────────────────────────────────────────────
import math, warnings
from datetime import date, datetime, timedelta
import pandas as pd
import yfinance as yf
warnings.filterwarnings('ignore')

# ─── Technical Indicators ────────────────────────────────────────────────────
def _ema(s, n):   return s.ewm(span=n, adjust=False).mean()

def _rsi(s, n=14):
    d = s.diff()
    g = d.clip(lower=0).ewm(com=n-1, adjust=False).mean()
    l = (-d.clip(upper=0)).ewm(com=n-1, adjust=False).mean()
    return 100 - 100 / (1 + g / l.replace(0, float('nan')))

def _macd(s, f=12, sl=26, sg=9):
    ml = _ema(s, f) - _ema(s, sl)
    si = _ema(ml, sg)
    return ml, si, ml - si

# ─── Data Fetching ───────────────────────────────────────────────────────────
def fetch_intraday(ticker, trading_date):
    kw = dict(tickers=ticker, interval='1m', progress=False, auto_adjust=True)
    if trading_date == date.today():
        df = yf.download(period='1d', **kw)
    else:
        end = trading_date + timedelta(days=1)
        df = yf.download(start=str(trading_date), end=str(end), **kw)
    if df.empty:
        raise ValueError(f'No intraday data for {ticker} on {trading_date}. '
                         'Market may be closed or ticker invalid.')
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    c = df['Close'].squeeze()
    df['EMA9'], df['EMA21'] = _ema(c, 9), _ema(c, 21)
    df['RSI'] = _rsi(c)
    df['MACD'], df['MACD_SIG'], df['MACD_HIST'] = _macd(c)
    return df

def fetch_prior_close(ticker, trading_date):
    h = yf.download(ticker, start=str(trading_date - timedelta(days=7)),
                    end=str(trading_date), interval='1d',
                    progress=False, auto_adjust=True)
    if isinstance(h.columns, pd.MultiIndex): h.columns = h.columns.get_level_values(0)
    return float(h['Close'].iloc[-1]) if not h.empty else float('nan')

def fetch_premarket(ticker, trading_date):
    try:
        end = trading_date + timedelta(days=1)
        df = yf.download(ticker, start=str(trading_date), end=str(end),
                         interval='1m', prepost=True, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
        pre = df[df.index.time < pd.Timestamp('09:30').time()]
        return (float(pre['High'].max()), float(pre['Low'].min())) if not pre.empty else (float('nan'), float('nan'))
    except Exception:
        return float('nan'), float('nan')

# ─── Analysis Engine ─────────────────────────────────────────────────────────
def analyse(ticker, df, prior_close, pm_high, pm_low):
    c        = df['Close'].squeeze()
    last     = float(c.iloc[-1])
    day_open = float(df['Open'].iloc[0])
    day_high = float(df['High'].max())
    day_low  = float(df['Low'].min())
    rng      = day_high - day_low
    pct      = (last - day_open) / day_open * 100
    e9, e21  = float(df['EMA9'].iloc[-1]), float(df['EMA21'].iloc[-1])
    rv       = float(df['RSI'].iloc[-1])
    mv, ms, mh = float(df['MACD'].iloc[-1]), float(df['MACD_SIG'].iloc[-1]), float(df['MACD_HIST'].iloc[-1])
    lv, av   = int(df['Volume'].iloc[-1]), int(df['Volume'].mean())
    vr       = lv / av if av else 1.0

    bear = sum([last < e9, last < e21, pct < 0, mh < 0, rv < 50])
    if   bear >= 4: bias, bc, bcc = ('Strongly Bearish' if bear==5 else 'Mildly Bearish'), '#f85149', 'bear'
    elif bear <= 1: bias, bc, bcc = ('Strongly Bullish' if bear==0 else 'Mildly Bullish'), '#3fb950', 'bull'
    else:           bias, bc, bcc = 'Neutral / Indecisive', '#d29922', 'neutral'

    ivm = (rng / day_open) / 0.012
    iv  = min(0.22 * ivm, 1.5)
    irl, irh = min(int(30*ivm), 95), min(int(30*ivm)+15, 99)
    ivr = 'Compressed' if irh<35 else 'Moderate' if irh<55 else 'Elevated' if irh<75 else 'Very High'
    ivc = '#3fb950' if ivr=='Compressed' else '#d29922' if ivr=='Moderate' else '#f85149'

    rs  = 1.0 if last >= 50 else 0.5
    nr  = round(last/rs)*rs
    mp  = round(((prior_close if not math.isnan(prior_close) else day_open) + day_open)/2/rs)*rs

    base = round(last/rs)*rs
    strikes = sorted({round((base+i*rs),2) for i in range(-5,6)})

    if bcc in ('bear','neutral') and irh >= 55:
        sn, st, sc2 = 'Bear Call Credit Spread', 'PRIMARY · High IV Premium Sell', 'primary'
        sell_c, buy_c = nr+rs, nr+3*rs
        clo, chi = round(rng*.12,2), round(rng*.18,2)
        srows = [('Sell',f'${sell_c:.2f} Call (0DTE)'),('Buy',f'${buy_c:.2f} Call (0DTE)'),
                 ('Net Credit (est.)',f'${clo:.2f} – ${chi:.2f}'),('Breakeven',f'${sell_c+chi:.2f}'),
                 ('Max Profit',f'Full credit if {ticker} stays ≤ ${sell_c:.2f}'),
                 ('Max Loss',f'${round(buy_c-sell_c-chi,2):.2f} / contract')]
        entry  = f'Enter now or on bounce to ${sell_c:.2f}'
        stopl  = f'Exit if price closes above ${sell_c+chi+.20:.2f}'
        tgt    = '75% of max credit, or let expire worthless'
        mr     = f'~${round((buy_c-sell_c-chi)*100):.0f}/contract'
        inv    = f'{ticker} closes above ${sell_c+chi:.2f}'
    elif bcc == 'bear':
        sn, st, sc2 = 'Bear Put Debit Spread', 'PRIMARY · Directional Bearish', 'primary'
        buy_p, sell_p = nr, nr-2*rs
        dlo, dhi = round(rng*.10,2), round(rng*.14,2)
        srows = [('Buy',f'${buy_p:.2f} Put (0DTE)'),('Sell',f'${sell_p:.2f} Put (0DTE)'),
                 ('Net Debit (est.)',f'${dlo:.2f} – ${dhi:.2f}'),('Breakeven',f'${buy_p-dhi:.2f}'),
                 ('Max Profit',f'${round(buy_p-sell_p-dhi,2):.2f} / contract'),
                 ('Max Loss',f'Debit paid (${dhi:.2f})')]
        entry = f'1-min close below ${day_low:.2f} (LOD breakdown)'
        stopl = f'Exit if price reclaims ${e9:.2f} (EMA 9)'
        tgt   = '70% of max profit'
        mr    = f'~${round(dhi*100):.0f}/contract'
        inv   = f'{ticker} reclaims ${e9:.2f}+'
    else:
        sn, st, sc2 = 'Bull Call Debit Spread', 'PRIMARY · Directional Bullish', 'bull-tag'
        buy_c2, sell_c2 = nr, nr+2*rs
        dlo, dhi = round(rng*.10,2), round(rng*.14,2)
        srows = [('Buy',f'${buy_c2:.2f} Call (0DTE)'),('Sell',f'${sell_c2:.2f} Call (0DTE)'),
                 ('Net Debit (est.)',f'${dlo:.2f} – ${dhi:.2f}'),('Breakeven',f'${buy_c2+dhi:.2f}'),
                 ('Max Profit',f'${round(sell_c2-buy_c2-dhi,2):.2f} / contract'),
                 ('Max Loss',f'Debit paid (${dhi:.2f})')]
        entry = f'1-min close above ${day_high:.2f} (HOD breakout)'
        stopl = f'Exit if price drops below ${e9:.2f} (EMA 9)'
        tgt   = '70% of max profit'
        mr    = f'~${round(dhi*100):.0f}/contract'
        inv   = f'{ticker} drops below ${e21:.2f}'

    ic_sc = round((last+rng*.35)/rs)*rs
    ic_bc = ic_sc+rs
    ic_sp = round((last-rng*.35)/rs)*rs
    ic_bp = ic_sp-rs

    rl = ('Oversold' if rv<30 else 'Near Oversold' if rv<35 else
          'Bearish'  if rv<50 else 'Neutral'       if rv<55 else
          'Bullish'  if rv<70 else 'Overbought')
    er = ('below both EMAs' if last<min(e9,e21) else
          'above both EMAs' if last>max(e9,e21) else 'between EMAs')

    return dict(
        ticker=ticker, last=last, day_open=day_open, day_high=day_high, day_low=day_low,
        rng=rng, pct=pct, prior_close=prior_close, pm_high=pm_high, pm_low=pm_low,
        e9=e9, e21=e21, rv=rv, rl=rl, rc=('bear' if rv<50 else 'bull'),
        mv=mv, ms=ms, mh=mh,
        ml=('Bearish Cross' if mh<0 else 'Bullish Cross'),
        mc=('bear' if mh<0 else 'bull'),
        lv=lv, av=av, vr=vr,
        bias=bias, bc=bc, bcc=bcc, er=er,
        iv=iv, irl=irl, irh=irh, ivr=ivr, ivc=ivc,
        mp=mp, strikes=strikes, rs=rs,
        ic_sc=ic_sc, ic_bc=ic_bc, ic_sp=ic_sp, ic_bp=ic_bp,
        ic_clo=round(rng*.08,2), ic_chi=round(rng*.12,2),
        sn=sn, st=st, sc2=sc2, srows=srows,
        entry=entry, stopl=stopl, tgt=tgt, mr=mr, inv=inv,
        trading_date=str(date.today()),
    )

# ─── HTML Builder ────────────────────────────────────────────────────────────
CSS = '''
*{box-sizing:border-box;margin:0;padding:0}
body{background:#0d1117;color:#c9d1d9;font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;font-size:15px;line-height:1.7;padding:32px 16px}
.container{max-width:980px;margin:0 auto}
header{background:linear-gradient(135deg,#161b22,#1f2937);border:1px solid #30363d;border-radius:12px;padding:28px 32px;margin-bottom:32px}
header h1{font-size:26px;font-weight:700;color:#58a6ff;margin-bottom:6px}
header .sub{color:#8b949e;font-size:14px}
.badge{display:inline-block;background:#21262d;border:1px solid #30363d;border-radius:20px;padding:3px 12px;font-size:12px;color:#8b949e;margin-top:10px;margin-right:8px}
.badge.bear{border-color:#f85149;color:#f85149}.badge.bull{border-color:#3fb950;color:#3fb950}.badge.neutral{border-color:#d29922;color:#d29922}
.section{background:#161b22;border:1px solid #30363d;border-radius:10px;padding:24px 28px;margin-bottom:24px}
.section h2{font-size:17px;font-weight:600;color:#e6edf3;margin-bottom:16px;padding-bottom:10px;border-bottom:1px solid #21262d;display:flex;align-items:center;gap:10px}
table{width:100%;border-collapse:collapse;font-size:14px;margin-top:4px}
th{background:#21262d;color:#8b949e;text-align:left;padding:9px 12px;font-weight:600;font-size:12px;text-transform:uppercase;letter-spacing:.04em}
td{padding:9px 12px;border-bottom:1px solid #21262d;color:#c9d1d9}
tr:last-child td{border-bottom:none}tr:hover td{background:#1c2128}td strong{color:#e6edf3}
.pg{display:grid;grid-template-columns:repeat(auto-fit,minmax(145px,1fr));gap:14px;margin-bottom:18px}
.pc{background:#21262d;border:1px solid #30363d;border-radius:8px;padding:14px 16px}
.pc .lb{font-size:11px;color:#8b949e;text-transform:uppercase;letter-spacing:.05em}
.pc .vl{font-size:22px;font-weight:700;color:#e6edf3;margin-top:4px}
.pc .vl.red{color:#f85149}.pc .vl.green{color:#3fb950}.pc .vl.blue{color:#58a6ff}.pc .sb{font-size:12px;color:#8b949e;margin-top:2px}
.bb{background:#1c2128;border-left:4px solid #f85149;border-radius:0 8px 8px 0;padding:14px 16px;margin-top:16px;font-size:14px}
.bb.neutral{border-left-color:#d29922}.bb.bull{border-left-color:#3fb950}
.sc{background:#1c2128;border:1px solid #30363d;border-radius:10px;padding:20px 22px;margin-bottom:18px}
.sc h3{font-size:15px;font-weight:700;color:#58a6ff;margin-bottom:4px}
.tag{display:inline-block;background:#21262d;border:1px solid #30363d;border-radius:4px;padding:2px 8px;font-size:11px;color:#8b949e;margin-bottom:14px}
.tag.primary{border-color:#f85149;color:#f85149}.tag.bull-tag{border-color:#3fb950;color:#3fb950}
.tc{display:grid;grid-template-columns:1fr 1fr;gap:18px}
.pill{display:inline-block;padding:2px 10px;border-radius:12px;font-size:12px;font-weight:600}
.pill.bear{background:rgba(248,81,73,.15);color:#f85149;border:1px solid rgba(248,81,73,.3)}
.pill.bull{background:rgba(63,185,80,.15);color:#3fb950;border:1px solid rgba(63,185,80,.3)}
.pill.neutral{background:rgba(210,153,34,.15);color:#d29922;border:1px solid rgba(210,153,34,.3)}
.sg{display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));gap:12px}
.si{background:#21262d;border-radius:8px;padding:14px 16px}
.si .sl{font-size:11px;color:#8b949e;text-transform:uppercase;letter-spacing:.05em}
.si .sv{font-size:15px;font-weight:600;color:#e6edf3;margin-top:4px}
.note{background:#1c2128;border:1px solid #d29922;border-radius:8px;padding:14px 16px;font-size:13px;color:#c9d1d9;margin-top:14px}
.note strong{color:#d29922}
.disc{background:#161b22;border:1px solid #30363d;border-radius:8px;padding:16px 20px;font-size:12px;color:#6e7681;margin-top:32px;text-align:center}
p{margin-bottom:10px;font-size:14px;color:#c9d1d9}
@media(max-width:640px){.tc{grid-template-columns:1fr}}
'''

def F(v, dec=2, pre='$'):  return 'N/A' if math.isnan(v) else f'{pre}{v:.{dec}f}'
def P(lbl, cls):           return f'<span class="pill {cls}">{lbl}</span>'
def card(lbl, val, vc, sub=''):
    return (f'<div class="pc"><div class="lb">{lbl}</div>'
            f'<div class="vl {vc}">{val}</div>'
            f'{"<div class=sb>"+sub+"</div>" if sub else ""}</div>')

def build_html(d, company):
    tk, lv = d['ticker'], d['last']
    cc = d['pct']
    chg = 'red' if cc < 0 else 'green'

    cards = []
    if not math.isnan(d['prior_close']): cards.append(card('Prior Close', F(d['prior_close']), ''))
    cards += [
        card('Last Price',    F(lv),              'blue'),
        card('Session Open',  F(d['day_open']),   ''),
        card('Day High',      F(d['day_high']),   'green'),
        card('Day Low',       F(d['day_low']),    'red'),
        card('Day Change',    f'{abs(cc):.2f}%',  chg, F(lv - d['day_open'])),
        card('Range',         F(d['rng']),        '', f"{d['rng']/d['day_open']*100:.2f}% of price"),
    ]
    if not math.isnan(d['pm_high']):
        cards += [card('Pre-Mkt High', F(d['pm_high']), 'green'),
                  card('Pre-Mkt Low',  F(d['pm_low']),  'red')]

    def lvr(price, name, role, cls, note):
        if math.isnan(price): return ''
        b  = '<strong>' if abs(price-lv)<0.05 else ''
        be = '</strong>' if b else ''
        return f'<tr><td>{name}</td><td>{b}{F(price)}{be}</td><td>{P(role,cls)}</td><td>{note}</td></tr>'

    chain = ''
    for s in sorted(d['strikes'], reverse=True):
        dist = s - lv
        if   dist > 0:                   ro, cl, nt = 'Call / OTM Resistance','bear','Above market'
        elif abs(dist) < d['rs']*.6:     ro, cl, nt = 'ATM Pin Zone','neutral','Max pain area'
        else:                             ro, cl, nt = 'Put / OTM Support','bull','Below market'
        if abs(s-d['mp']) < d['rs']*.6:  nt += ' · Max Pain'
        chain += (f'<tr><td><strong>${s:.2f}</strong></td>'
                  f'<td>{"Call" if dist>=0 else "Put"}</td>'
                  f'<td>{P(ro,cl)}</td><td>{nt}</td></tr>')

    sr = ''.join(f'<tr><td>{r[0]}</td><td><strong>{r[1]}</strong></td></tr>' for r in d['srows'])

    e9c  = 'bear' if lv < d['e9']  else 'bull'
    e21c = 'bear' if lv < d['e21'] else 'bull'
    now  = datetime.now().strftime('%b %d, %Y  %H:%M')

    ivfav = d['irh'] >= 55
    pcr_txt = ('Likely &gt;1.2 (put-heavy)' if d['bcc']=='bear'
               else 'Likely &lt;0.8 (call-heavy)' if d['bcc']=='bull'
               else 'Likely ~1.0 (balanced)')

    html = f'''<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1.0"/>
<title>{tk} 0DTE Options — {d['trading_date']}</title>
<style>{CSS}</style></head>
<body><div class="container">

<header>
  <h1>{tk} &middot; 0DTE Options Trading Setup</h1>
  <div class="sub">{company} &nbsp;|&nbsp; {d['trading_date']} &nbsp;|&nbsp; Generated {now}</div>
  <div style="margin-top:12px">
    <span class="badge {d['bcc']}">{d['bias']}</span>
    <span class="badge">0DTE</span>
    <span class="badge">yfinance &middot; Live Data</span>
  </div>
</header>

<div class="section">
  <h2>&#x1F4C8; 1. Current Price &amp; Intraday Trend</h2>
  <div class="pg">{''.join(cards)}</div>
  <table>
    <tr><th>Indicator</th><th>Value</th><th>Signal</th></tr>
    <tr><td>EMA 9</td><td><strong>{F(d['e9'])}</strong></td><td>{P('Resistance' if lv<d['e9'] else 'Support', e9c)}</td></tr>
    <tr><td>EMA 21</td><td><strong>{F(d['e21'])}</strong></td><td>{P('Resistance' if lv<d['e21'] else 'Support', e21c)}</td></tr>
    <tr><td>RSI (14)</td><td><strong>{d['rv']:.2f}</strong></td><td>{P(d['rl'], d['rc'])}</td></tr>
    <tr><td>MACD</td><td><strong>{d['mv']:.4f} vs Signal {d['ms']:.4f}</strong></td><td>{P(d['ml'], d['mc'])}</td></tr>
    <tr><td>MACD Histogram</td><td><strong>{d['mh']:.4f}</strong></td>
        <td>{P('Negative — Bearish' if d['mh']<0 else 'Positive — Bullish', d['mc'])}</td></tr>
    <tr><td>Volume (last bar)</td><td><strong>{d['lv']:,}</strong></td>
        <td>{P(f"{d['vr']:.1f}\u00d7 avg", d['bcc'])}</td></tr>
  </table>
  <div class="bb {d['bcc']}">
    <strong>Bias: {d['bias']}.</strong> {tk} opened at {F(d['day_open'])}, high {F(d['day_high'])}, low {F(d['day_low'])}.
    Price ({F(lv)}) is {d['er']}. RSI {d['rv']:.1f} is {d['rl'].lower()}.
    MACD histogram {'negative &mdash; bearish' if d['mh']<0 else 'positive &mdash; bullish'}.
  </div>
</div>

<div class="section">
  <h2>&#x1F5FA; 2. Key Support &amp; Resistance Levels</h2>
  <table>
    <tr><th>Level</th><th>Price</th><th>Type</th><th>Notes</th></tr>
    {lvr(d['day_high'],   'Intraday High',  'Resistance', 'bear', "Today's HOD")}
    {lvr(d['day_open'],   'Session Open',   'Resistance' if lv<d['day_open'] else 'Support', 'bear' if lv<d['day_open'] else 'bull', 'Gap reference')}
    {lvr(d['pm_high'],    'Pre-Mkt High',   'Resistance', 'bear', 'Pre-market HOD') if not math.isnan(d['pm_high']) else ''}
    {lvr(d['e21'],        'EMA 21',         'Resistance' if lv<d['e21'] else 'Support', e21c, 'Slower dynamic EMA')}
    {lvr(d['e9'],         'EMA 9',          'Resistance' if lv<d['e9']  else 'Support', e9c,  'Faster dynamic EMA')}
    <tr style="background:#1c2128"><td><strong>CURRENT PRICE</strong></td>
      <td><strong style="color:#58a6ff">{F(lv)}</strong></td>
      <td>{P('Live Quote','neutral')}</td><td>{d['er']}</td></tr>
    {lvr(d['pm_low'],     'Pre-Mkt Low',    'Support', 'bull', 'Pre-market LOD') if not math.isnan(d.get('pm_low', float('nan'))) else ''}
    {lvr(d['day_low'],    'Intraday Low',   'Support', 'bull', "Today's LOD")}
    {lvr(d['prior_close'],'Prior Close',    'Support', 'bull', 'Previous session close') if not math.isnan(d['prior_close']) else ''}
  </table>
</div>

<div class="section">
  <h2>&#x1F321; 3. Implied Volatility &amp; IV Rank</h2>
  <p>Live IV requires a broker API. Estimated from today&#39;s intraday range:</p>
  <table>
    <tr><th>Metric</th><th>Observation</th><th>Implication</th></tr>
    <tr><td>Intraday Range</td>
        <td><strong>{F(d['rng'])} ({d['rng']/d['day_open']*100:.2f}%)</strong></td>
        <td>{'Elevated' if d['rng']/d['day_open']>.015 else 'Normal'}</td></tr>
    <tr><td>Volume Spike</td>
        <td><strong>{d['lv']:,} vs {d['av']:,} avg ({d['vr']:.1f}&times;)</strong></td>
        <td>{'Institutional / event-driven' if d['vr']>3 else 'Normal volume'}</td></tr>
    <tr><td>Estimated IV</td><td><strong>~{d['iv']*100:.0f}%</strong></td>
        <td style="color:{d['ivc']}">{d['ivr']}</td></tr>
    <tr><td>IV Rank Estimate</td><td><strong>~{d['irl']}&#8211;{d['irh']}th percentile</strong></td>
        <td>{'Premium selling favoured' if ivfav else 'Debit spreads favoured'}</td></tr>
  </table>
  <div class="note"><strong>&#x26A0; Verify on broker platform</strong> for precise IV / IVR values.</div>
</div>

<div class="section">
  <h2>&#x1F4CA; 4. Put/Call Ratio &amp; Sentiment</h2>
  <p>Live PCR requires broker / CBOE data. Sentiment inferred from price action:</p>
  <table>
    <tr><th>Signal</th><th>Observation</th></tr>
    <tr><td>Price vs. Open</td><td>{cc:.2f}% &mdash; {'sellers in control' if cc<0 else 'buyers in control'}</td></tr>
    <tr><td>RSI Level</td><td>{d['rv']:.1f} &mdash; {d['rl']}</td></tr>
    <tr><td>Implied PCR</td><td>{pcr_txt}</td></tr>
    <tr><td>Overall Sentiment</td><td>{P(d['bias'], d['bcc'])}</td></tr>
  </table>
  <div class="note"><strong>&#x26A0; Verify:</strong> Check CBOE or broker for {tk}-specific PCR.
    PCR &gt;1.5 = extreme fear &nbsp;|&nbsp; PCR &lt;0.7 = greed.</div>
</div>

<div class="section">
  <h2>&#x1F517; 5. Options Chain Snapshot (0DTE &mdash; {d['trading_date']})</h2>
  <table><tr><th>Strike</th><th>Type</th><th>Role</th><th>Notes</th></tr>{chain}</table>
  <div class="bb neutral" style="margin-top:16px">
    <strong>Max Pain Estimate: ~{F(d['mp'])}.</strong>
    Current price ({F(lv)}) is {'below' if lv<d['mp'] else 'above'} max pain &mdash;
    expect {'upward' if lv<d['mp'] else 'downward'} gravitational pull into close.
    Verify on your broker&#39;s options chain.
  </div>
</div>

<div class="section">
  <h2>&#x26A1; 6. Strategy Recommendation</h2>
  <div class="sc">
    <h3>Strategy A &mdash; {d['sn']}</h3>
    <span class="tag {d['sc2']}">{d['st']}</span>
    <table><tr><th>Parameter</th><th>Value</th></tr>{sr}</table>
  </div>
  <div class="sc">
    <h3>Strategy B &mdash; Iron Condor</h3>
    <span class="tag">SECONDARY &middot; Range / Theta Play</span>
    <table>
      <tr><th>Wing</th><th>Sell</th><th>Buy</th></tr>
      <tr><td>Upper (Call)</td><td><strong>${d['ic_sc']:.2f} Call</strong></td><td><strong>${d['ic_bc']:.2f} Call</strong></td></tr>
      <tr><td>Lower (Put)</td><td><strong>${d['ic_sp']:.2f} Put</strong></td><td><strong>${d['ic_bp']:.2f} Put</strong></td></tr>
    </table>
    <table style="margin-top:10px">
      <tr><th>Parameter</th><th>Value</th></tr>
      <tr><td>Net Credit (est.)</td><td><strong>${d['ic_clo']:.2f} &ndash; ${d['ic_chi']:.2f}</strong></td></tr>
      <tr><td>Max Profit Zone</td><td><strong>${d['ic_sp']:.2f} &ndash; ${d['ic_sc']:.2f} at expiry</strong></td></tr>
      <tr><td>Max Loss</td><td><strong>~${round(d['rs']-d['ic_chi'],2):.2f}/contract</strong></td></tr>
    </table>
  </div>
</div>

<div class="section">
  <h2>&#x1F3AF; 7. Entry, Target &amp; Stop-Loss</h2>
  <div class="tc">
    <div>
      <h3 style="font-size:14px;color:#58a6ff;margin-bottom:12px">Strategy A &mdash; {d['sn']}</h3>
      <table>
        <tr><th>Parameter</th><th>Value</th></tr>
        <tr><td>Entry Trigger</td><td><strong>{d['entry']}</strong></td></tr>
        <tr><td>Profit Target</td><td>{d['tgt']}</td></tr>
        <tr><td>Stop-Loss</td><td>{d['stopl']}</td></tr>
        <tr><td>Time Stop</td><td>Exit by <strong>3:00 PM ET</strong></td></tr>
      </table>
    </div>
    <div>
      <h3 style="font-size:14px;color:#58a6ff;margin-bottom:12px">Strategy B &mdash; Iron Condor</h3>
      <table>
        <tr><th>Parameter</th><th>Value</th></tr>
        <tr><td>Entry Window</td><td><strong>11:00 AM &ndash; 12:30 PM ET</strong></td></tr>
        <tr><td>Profit Target</td><td>50% of max credit</td></tr>
        <tr><td>Stop-Loss</td><td>Exit if price breaches ${d['ic_sc']:.2f} or ${d['ic_sp']:.2f}</td></tr>
        <tr><td>Time Stop</td><td>Let expire if inside profit zone at 3:45 PM ET</td></tr>
      </table>
    </div>
  </div>
</div>

<div class="section">
  <h2>&#x26A0;&#xFE0F; 8. Risk &amp; Invalidation</h2>
  <div class="tc">
    <div>
      <h3 style="font-size:14px;color:#f85149;margin-bottom:12px">Strategy A</h3>
      <table>
        <tr><th>Risk Factor</th><th>Detail</th></tr>
        <tr><td>Max Loss</td><td><strong>{d['mr']} per contract</strong></td></tr>
        <tr><td>Invalidation</td><td>{d['inv']}</td></tr>
        <tr><td>IV Risk</td><td>{'IV crush helps (short premium)' if ivfav else 'IV expansion hurts (long premium)'}</td></tr>
        <tr><td>Time Decay</td><td>{'Theta works for you' if ivfav else 'Exit before 3 PM ET'}</td></tr>
        <tr><td>News Risk</td><td>Unexpected catalyst can gap through strikes</td></tr>
      </table>
    </div>
    <div>
      <h3 style="font-size:14px;color:#f85149;margin-bottom:12px">Strategy B &mdash; Iron Condor</h3>
      <table>
        <tr><th>Risk Factor</th><th>Detail</th></tr>
        <tr><td>Max Loss</td><td><strong>~${round(d['rs']-d['ic_chi'],2):.2f}/contract if wing breached</strong></td></tr>
        <tr><td>Invalidation</td><td>Move outside ${d['ic_sp']:.2f}&ndash;${d['ic_sc']:.2f}</td></tr>
        <tr><td>Macro Risk</td><td>Fed / data release could spike vol</td></tr>
        <tr><td>Assignment Risk</td><td>Exit by 3:45 PM ET</td></tr>
      </table>
    </div>
  </div>
</div>

<div class="section">
  <h2>&#x1F4CB; Trade Summary</h2>
  <div class="sg">
    <div class="si"><div class="sl">Ticker</div><div class="sv" style="color:#58a6ff">{tk}</div></div>
    <div class="si"><div class="sl">Bias</div><div class="sv" style="color:{d['bc']}">{d['bias']}</div></div>
    <div class="si"><div class="sl">Last Price</div><div class="sv">{F(lv)}</div></div>
    <div class="si"><div class="sl">Primary Trade</div><div class="sv">{d['sn']}</div></div>
    <div class="si"><div class="sl">Max Risk (Strat A)</div><div class="sv" style="color:#f85149">{d['mr']}</div></div>
    <div class="si"><div class="sl">IV Regime</div><div class="sv" style="color:{d['ivc']}">{d['ivr']} (~{d['irl']}&ndash;{d['irh']}th pct)</div></div>
    <div class="si"><div class="sl">Invalidation</div><div class="sv">{d['inv']}</div></div>
    <div class="si"><div class="sl">Expiry</div><div class="sv">0DTE &mdash; {d['trading_date']}</div></div>
  </div>
</div>

<div class="disc">This report is generated from public market data (yfinance) and is for educational purposes only.
  IV, PCR, max pain, and exact options chain data must be verified on your broker platform before placing any trade.
  Options trading involves significant risk of loss. <strong>This is not financial advice.</strong></div>
</div></body></html>'''
    return html

print('✅ All functions loaded — run Cell 3 to generate a report')

In [ ]:
# ── CELL 3 · Generate Report ──────────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, HTML, FileLink, clear_output

ticker_box = widgets.Text(
    value='MSFT', placeholder='MSFT, CSCO, NVDA …',
    description='Ticker:', layout=widgets.Layout(width='260px')
)
date_box = widgets.Text(
    value=str(date.today()), placeholder='YYYY-MM-DD',
    description='Date:', layout=widgets.Layout(width='220px')
)
btn = widgets.Button(
    description='⚡ Generate Report', button_style='primary',
    layout=widgets.Layout(width='180px', height='36px')
)
out = widgets.Output()
display(widgets.HBox([ticker_box, date_box, btn]))
display(out)

def run(_):
    with out:
        clear_output(wait=True)
        tk = ticker_box.value.strip().upper()
        try:
            td = date.fromisoformat(date_box.value.strip())
        except ValueError:
            print(f'❌ Invalid date. Use YYYY-MM-DD format.')
            return

        print(f'⏳ [{tk}] Fetching 1-min intraday data for {td}…')
        try:
            df = fetch_intraday(tk, td)
        except ValueError as e:
            print(f'❌ {e}'); return

        print(f'⏳ [{tk}] Fetching prior close & pre-market…')
        pc       = fetch_prior_close(tk, td)
        pmh, pml = fetch_premarket(tk, td)

        print(f'⏳ [{tk}] Running analysis…')
        d = analyse(tk, df, pc, pmh, pml)

        try:
            company = yf.Ticker(tk).info.get('longName', tk)
        except Exception:
            company = tk

        html_str = build_html(d, company)
        fname    = f'{tk}_options_{td}.html'
        with open(fname, 'w', encoding='utf-8') as f:
            f.write(html_str)

        print(f'\n✅ Report ready for {tk} ({company})')
        print(f'   Bias      : {d["bias"]}')
        print(f'   Last      : ${d["last"]:.2f}  |  Day change: {d["pct"]:+.2f}%')
        print(f'   Strategy A: {d["sn"]}')
        print(f'   Max Risk  : {d["mr"]}')
        print(f'   IV Regime : {d["ivr"]} (~{d["irl"]}–{d["irh"]}th pct)')
        print()

        # ── Download ──────────────────────────────────────────────────────────
        try:
            from google.colab import files
            print('📥 Triggering download…')
            files.download(fname)
        except ImportError:
            display(FileLink(fname, result_html_prefix='📥 Download: '))

        # ── Inline preview ────────────────────────────────────────────────────
        import base64
        b64 = base64.b64encode(html_str.encode()).decode()
        display(HTML(
            f'<p style="margin-top:16px;color:#8b949e;font-size:13px">'
            f'👇 Inline preview (scroll within frame)</p>'
            f'<iframe src="data:text/html;base64,{b64}" '
            f'width="100%" height="950" '
            f'style="border:1px solid #30363d;border-radius:8px;margin-top:4px">'
            f'</iframe>'
        ))

btn.on_click(run)